# Chapter 12: Simpson's paradox

In [1]:
import numpy as np
from expkit.sim.user_segments import SegmentSpec, imbalanced_assignment
from expkit.plot.style import apply_style
apply_style()

## Loop A: build the paradox

In [2]:
specs = [SegmentSpec('A', 0.20, 0.80, 0.05), SegmentSpec('B', 0.80, 0.20, 0.05)]
share = {'A': 0.20, 'B': 0.80}
pop = imbalanced_assignment(20000, specs, share, seed=1200)
for s in specs:
    c = pop[s.name]['control'].mean(); t = pop[s.name]['treatment'].mean()
    print(f'segment {s.name}:  control={c:.3f}  treatment={t:.3f}  diff={t-c:+.3f}')
c_total = np.concatenate([pop[s.name]['control'] for s in specs])
t_total = np.concatenate([pop[s.name]['treatment'] for s in specs])
print(f'AGGREGATE:        control={c_total.mean():.3f}  treatment={t_total.mean():.3f}  diff={t_total.mean()-c_total.mean():+.3f}')

segment A:  control=0.794  treatment=0.851  diff=+0.057
segment B:  control=0.198  treatment=0.249  diff=+0.051
AGGREGATE:        control=0.496  treatment=0.284  diff=-0.212


## Loop C: balanced vs imbalanced

In [3]:
rng = np.random.default_rng(0)
for label, sa in [('balanced 50/50', 0.5), ('imbalanced 20/80', 0.2)]:
    runs = []
    for _ in range(30):
        share = {'A': sa, 'B': 1 - sa}
        pop = imbalanced_assignment(10000, specs, share, seed=int(rng.integers(0, 2**30)))
        c = np.concatenate([pop[s.name]['control'] for s in specs])
        t = np.concatenate([pop[s.name]['treatment'] for s in specs])
        runs.append(t.mean() - c.mean())
    print(f'{label:<22} mean aggregate effect = {np.mean(runs):+.3f}  std = {np.std(runs):.3f}')

balanced 50/50         mean aggregate effect = +0.050  std = 0.007
imbalanced 20/80       mean aggregate effect = -0.212  std = 0.009
